# ChashiBondhu AI: Climate Data Preparation (Phase 1)

**Goal:** turn your 31,200-row NASA POWER / MERRA-2 grid file into **district-level, crop-season climate features** that can feed the Climate-Aware Crop Selector and the Planting-Window Comparison.

**Steps in this notebook**

| Step | What it does |
|---|---|
| 0 | Install/import libraries |
| 1 | Load and validate your CSV |
| 2 | Look at the grid (and see why polygon matching alone fails) |
| 3 | Load official district (Zila) boundaries |
| 4 | Map grid points to districts (distance-weighted) |
| 5 | Monthly features: rainfall mm, DTR |
| 6 | Crop calendar configuration |
| 7 | Seasonal aggregation per district and crop |
| 8 | Merge BBS yield data (when you have it) |
| 9 | Climatology tables for the website and quick plots |
| 10 | Save and download outputs |

**Honest limits to remember in your pitch:** the data is a ~55 km x 70 km model grid, so district values are *regional estimates*, not farm-level or station measurements. Monthly data cannot show daily extremes. Every threshold below is a starting assumption that you should replace with a cited value from BRRI / BARI / DAE / FAO EcoCrop.

## Step 0: Setup

Colab already has most libraries. `geopandas` is installed only if missing.

In [ ]:
import importlib, subprocess, sys, os, io, zipfile

for pkg in ["geopandas", "requests"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.neighbors import BallTree

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
print("Libraries ready")

## Step 1: Load and validate the CSV

Run this cell and choose `Bangladesh_Climate_Master_Fixed_2001_2025.csv` when Colab asks. (If the file is already in the Colab file panel, it is not uploaded again.)

**Units in your file (NASA POWER monthly):**
- `T2M`, `T2M_MAX`, `T2M_MIN`: degrees C
- `PRECTOTCORR`: **mm per day** (monthly average). Multiply by days in month for monthly total
- `RH2M`: %
- `SOLAR`: MJ per m2 per day

In [ ]:
DATA_FILE = "Bangladesh_Climate_Master_Fixed_2001_2025.csv"

if not os.path.exists(DATA_FILE):
    from google.colab import files
    uploaded = files.upload()          # pick the CSV in the dialog
    DATA_FILE = list(uploaded.keys())[0]

df = pd.read_csv(DATA_FILE)
print("Shape:", df.shape)
df.head()

In [ ]:
# --- Validation checks -------------------------------------------------
n_cells = df[["LAT", "LON"]].drop_duplicates().shape[0]
n_years = df["YEAR"].nunique()
expected = n_cells * n_years * 12

print(f"Grid cells: {n_cells} | Years: {n_years} | Expected rows: {expected} | Actual rows: {len(df)}")
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate (YEAR, MO, LAT, LON) keys:", int(df.duplicated(["YEAR", "MO", "LAT", "LON"]).sum()))
print("NASA fill values (-999):", int((df.select_dtypes("number") == -999).sum().sum()))

# Logical consistency: Tmin <= Tmean <= Tmax
bad = df[~((df.T2M_MIN <= df.T2M) & (df.T2M <= df.T2M_MAX))]
print("Rows where Tmin <= T2M <= Tmax is violated:", len(bad))

print("\nValue ranges:")
display(df[["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "SOLAR"]].describe().T[["min", "mean", "max"]])

In [ ]:
# --- Extreme values worth a second look ---------------------------------
hot = df[df.T2M_MAX > 45]
cold = df[df.T2M_MIN < 3]
print(f"Rows with T2M_MAX > 45 C: {len(hot)} (years: {sorted(int(y) for y in hot.YEAR.unique())})")
print(f"Rows with T2M_MIN < 3 C : {len(cold)} (years: {sorted(int(y) for y in cold.YEAR.unique())})")
print("Longitude range of the hot rows:", hot.LON.min(), "to", hot.LON.max())

**How to read the check above:** very high `T2M_MAX` in April to June and very low `T2M_MIN` in January 2013 may be real events (heatwaves, the 2013 cold wave) that a grid model smooths or exaggerates. They mostly sit on the western grid columns, which include West Bengal (India). Before relying on them, compare a few values with Bangladesh Meteorological Department (BMD) station records, and confirm whether `T2M_MAX` in your download is the *monthly mean of daily maximum* (what we assume) or the *absolute monthly maximum*.

## Step 2: Look at the grid

The file has **104 grid points**, spaced 0.5 degrees north-south and 0.625 degrees east-west. That is far fewer than the 64 districts, and some points fall over India or the Bay of Bengal.

In [ ]:
grid = df[["LAT", "LON"]].drop_duplicates().sort_values(["LAT", "LON"]).reset_index(drop=True)
grid["grid_id"] = grid.index

fig, ax = plt.subplots(figsize=(5.5, 6.5))
ax.scatter(grid.LON, grid.LAT, s=25)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("104 climate grid points")
ax.grid(alpha=.3)
plt.show()

## Step 3: District boundaries (Zila)

We use the free **GADM 4.1** boundaries (level 2 = districts). If the download is blocked, download any Bangladesh district GeoJSON or shapefile yourself, upload it to Colab, and set `DISTRICT_FILE` to its name. Then adjust the `rename` line so the district and division column names match.

For each district we compute a **representative point** (guaranteed to be inside the polygon, even for odd shapes like coastal or riverine districts).

In [ ]:
import requests

DISTRICT_FILE = "gadm41_BGD_2.json"
GADM_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_BGD_2.json.zip"

if not os.path.exists(DISTRICT_FILE):
    r = requests.get(GADM_URL, timeout=120)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extractall(".")

districts = gpd.read_file(DISTRICT_FILE)
districts = districts[["NAME_1", "NAME_2", "geometry"]].rename(
    columns={"NAME_1": "division", "NAME_2": "district"})
districts = districts.to_crs("EPSG:4326").reset_index(drop=True)

rep = districts.geometry.representative_point()
districts["lat"] = rep.y
districts["lon"] = rep.x

print("Districts loaded:", len(districts), "(Bangladesh has 64; check any extras such as lakes)")
districts.drop(columns="geometry").head(10)

### Why not just use "point inside polygon"?

Your plan was to map grid points to districts with polygons. With a 0.5 x 0.625 degree grid, most districts contain **no** grid point at all. The check below proves it, and motivates the next step.

In [ ]:
grid_gdf = gpd.GeoDataFrame(grid, geometry=gpd.points_from_xy(grid.LON, grid.LAT), crs="EPSG:4326")
inside = gpd.sjoin(grid_gdf, districts[["district", "geometry"]], predicate="within")

covered = inside["district"].nunique()
print(f"Grid points inside Bangladesh: {len(inside)} of {len(grid)}")
print(f"Districts containing at least one grid point: {covered} of {len(districts)}")
print(f"Districts with NO grid point: {len(districts) - covered}")

## Step 4: Map grid points to districts (inverse-distance weighting)

**Method:** for each district's representative point, find the **4 nearest grid points** and average them with weights `1 / distance^2`. Closer grid points count more.

- Uses a haversine `BallTree` so distances are real kilometres on the globe.
- Every district gets a value, including those with no grid point inside.
- Because the averaging smooths values, it will understate local extremes. State this as a limitation.

In [ ]:
K = 4          # nearest grid points per district
POWER = 2      # inverse-distance power

tree = BallTree(np.deg2rad(grid[["LAT", "LON"]].values), metric="haversine")
dist_rad, idx = tree.query(np.deg2rad(districts[["lat", "lon"]].values), k=K)
dist_km = dist_rad * 6371.0

w = 1.0 / np.maximum(dist_km, 1.0) ** POWER          # avoid divide-by-zero
w = w / w.sum(axis=1, keepdims=True)                 # weights sum to 1 per district

mapping = pd.DataFrame({
    "district": np.repeat(districts["district"].values, K),
    "division": np.repeat(districts["division"].values, K),
    "grid_id":  idx.ravel(),
    "dist_km":  dist_km.ravel(),
    "weight":   w.ravel(),
})

print("Nearest-point distance (km): median =", round(np.median(dist_km[:, 0]), 1),
      "| max =", round(dist_km[:, 0].max(), 1))
mapping.head(8)

In [ ]:
VARS = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "SOLAR"]

df = df.merge(grid, on=["LAT", "LON"])                       # add grid_id
m = mapping.merge(df[["grid_id", "YEAR", "MO"] + VARS], on="grid_id")

# weighted average of every variable
for v in VARS:
    m[v] = m[v] * m["weight"]

dm = (m.groupby(["division", "district", "YEAR", "MO"], as_index=False)[VARS].sum())

print("District-month table:", dm.shape,
      "| expected:", len(districts), "x", n_years, "x 12 =", len(districts) * n_years * 12)
print("Missing values:", int(dm.isna().sum().sum()))
dm.head()

## Step 5: Monthly agronomic features

- **Monthly rainfall (mm)** = `PRECTOTCORR` (mm/day) x days in that month
- **DTR** (diurnal temperature range) = `T2M_MAX - T2M_MIN`. A large DTR in grain filling is generally favourable, a very small one often means cloudy, humid weather

In [ ]:
dm["days"] = pd.to_datetime(dict(year=dm.YEAR, month=dm.MO, day=1)).dt.days_in_month
dm["RAIN_MM"] = dm["PRECTOTCORR"] * dm["days"]
dm["DTR"] = dm["T2M_MAX"] - dm["T2M_MIN"]

dm[["district", "YEAR", "MO", "T2M", "RAIN_MM", "DTR"]].head()

## Step 6: Crop calendar and stress thresholds

Each crop has:
- `start`, `length`: growing season (month number and number of months). A season can cross the new year (Boro starts in December)
- `sensitive`: months of the sensitive stage (for example flowering) that get their own features
- `tmax_heat`: a month counts as a **heat-stress month** if mean daily max exceeds this (deg C)
- `tmin_cold`: a month counts as a **cold-stress month** if mean daily min falls below this (deg C)
- `dry_mm`, `wet_mm`: months below or above this monthly rainfall are counted as dry or excessively wet
- `disease`: key of the disease-risk rule used below

> **These numbers are starter assumptions, not verified agronomy.** Replace each with a value from BRRI, BARI, DAE or FAO EcoCrop and note the source in your documentation. Judges will check this.

In [ ]:
CROPS = {
    #                start len  sensitive   heat  cold  dry  wet  disease
    "aman_rice":    dict(start=6,  length=6, sensitive=[9, 10], tmax_heat=35, tmin_cold=15, dry_mm=100, wet_mm=600, disease="humid_warm"),
    "aus_rice":     dict(start=3,  length=5, sensitive=[6, 7],  tmax_heat=35, tmin_cold=15, dry_mm=100, wet_mm=600, disease="humid_warm"),
    "boro_rice":    dict(start=12, length=6, sensitive=[3, 4],  tmax_heat=35, tmin_cold=15, dry_mm=20,  wet_mm=250, disease="humid_warm"),
    "wheat":        dict(start=11, length=5, sensitive=[2, 3],  tmax_heat=30, tmin_cold=5,  dry_mm=10,  wet_mm=150, disease=None),
    "potato":       dict(start=11, length=4, sensitive=[1, 2],  tmax_heat=30, tmin_cold=5,  dry_mm=10,  wet_mm=150, disease="late_blight"),
    "maize_rabi":   dict(start=11, length=6, sensitive=[2, 3],  tmax_heat=35, tmin_cold=8,  dry_mm=10,  wet_mm=200, disease=None),
    "mustard":      dict(start=11, length=4, sensitive=[12, 1], tmax_heat=30, tmin_cold=5,  dry_mm=10,  wet_mm=100, disease=None),
    "jute":         dict(start=3,  length=5, sensitive=[5, 6],  tmax_heat=38, tmin_cold=15, dry_mm=100, wet_mm=500, disease=None),
}

# Disease risk rules, evaluated on monthly means (so they are risk INDICATORS, not predictions)
DISEASE_RULES = {
    # Late blight likes cool and humid weather, roughly 10-25 C with high humidity.
    "late_blight": lambda d: (d.RH2M >= 85) & (d.T2M.between(10, 25)),
    # Generic humid and warm months for rice (blast, sheath blight). Not a specific disease model.
    "humid_warm":  lambda d: (d.RH2M >= 85) & (d.T2M.between(25, 32)),
}
print("Configured crops:", list(CROPS))

**Domain note on potato blight:** you described the indicator as "high RH + warm temperature". Late blight in potato is actually favoured by **cool, humid** weather (around 10-25 degrees C), which is why the rule above uses that range. Warm-humid conditions matter more for diseases such as rice blast. Confirm the exact ranges with a BARI source and cite it.

## Step 7: Seasonal aggregation per district and crop

For each crop we:
1. keep only the months of its growing season,
2. assign each month to a **season year** = the year the season **starts** (so Boro Dec 2018 to May 2019 is `season_year = 2018`),
3. drop incomplete seasons (for example a season that started in December 2025 has no data yet),
4. compute totals, means, stress-month counters and sensitive-window features.

`window_tmin_lowest` is the lowest **monthly mean of daily minimum** in the sensitive window. Monthly data cannot give the absolute coldest night, so do not call it "minimum temperature on the coldest day".

In [ ]:
def season_features(crop, p):
    # (month, offset) pairs: offset 1 means the month falls in the next calendar year
    months = [((p["start"] - 1 + i) % 12 + 1, (p["start"] - 1 + i) // 12) for i in range(p["length"])]
    mo2off = dict(months)

    s = dm[dm.MO.isin(mo2off)].copy()
    s["season_year"] = s["YEAR"] - s["MO"].map(mo2off)

    # month-level flags
    s["heat_m"] = (s.T2M_MAX > p["tmax_heat"]).astype(int)
    s["cold_m"] = (s.T2M_MIN < p["tmin_cold"]).astype(int)
    s["dry_m"]  = (s.RAIN_MM < p["dry_mm"]).astype(int)
    s["wet_m"]  = (s.RAIN_MM > p["wet_mm"]).astype(int)
    s["disease_m"] = DISEASE_RULES[p["disease"]](s).astype(int) if p["disease"] else 0

    keys = ["division", "district", "season_year"]
    out = s.groupby(keys).agg(
        n_months=("MO", "size"),
        season_rain_mm=("RAIN_MM", "sum"),
        season_tmean=("T2M", "mean"),
        season_tmax=("T2M_MAX", "mean"),
        season_tmin=("T2M_MIN", "mean"),
        season_dtr=("DTR", "mean"),
        season_rh=("RH2M", "mean"),
        season_solar=("SOLAR", "mean"),
        heat_stress_months=("heat_m", "sum"),
        cold_stress_months=("cold_m", "sum"),
        dry_months=("dry_m", "sum"),
        wet_months=("wet_m", "sum"),
        disease_risk_months=("disease_m", "sum"),
    ).reset_index()

    # sensitive-window features (e.g. flowering)
    w = s[s.MO.isin(p["sensitive"])]
    wf = w.groupby(keys).agg(
        window_rain_mm=("RAIN_MM", "sum"),
        window_tmean=("T2M", "mean"),
        window_tmax_highest=("T2M_MAX", "max"),
        window_tmin_lowest=("T2M_MIN", "min"),
        window_dtr=("DTR", "mean"),
    ).reset_index()

    out = out.merge(wf, on=keys, how="left")
    out = out[out.n_months == p["length"]].drop(columns="n_months")   # keep complete seasons only
    out.insert(0, "crop", crop)
    return out

season_df = pd.concat([season_features(c, p) for c, p in CROPS.items()], ignore_index=True)

print("Season feature table:", season_df.shape)
print(season_df.groupby("crop").size())
season_df.head()

In [ ]:
# Sanity check: Aman rice (June-November) season rainfall should be high everywhere
aman = season_df[season_df.crop == "aman_rice"]
print(aman.groupby("district").season_rain_mm.mean().sort_values().round(0).iloc[[0, 1, 2, -3, -2, -1]])

## Step 8: Merge BBS yield data (target variable)

You still need to collect district-level crop **area and production** from the Bangladesh Bureau of Statistics (Yearbook of Agricultural Statistics / district-level tables). Save them as a CSV named `bbs_yield.csv` with these columns:

| district | season_year | crop | area | production |
|---|---|---|---|---|
| Dhaka | 2018 | aman_rice | 12345 | 30000 |

Rules for filling it in:
- `crop` must use the keys from `CROPS` above (`aman_rice`, `boro_rice`, `wheat` ...)
- `season_year` is the **first year** of the BBS agricultural year (BBS "2018-19" means `2018`). Check this against the season definitions in Step 6
- Check units. BBS tables often use **acres** for area. If so, set `AREA_UNIT = "acre"` below so yield comes out in tonnes per hectare

If the file is not there yet, the cell simply continues without yields.

In [ ]:
import re

AREA_UNIT = "ha"                    # change to "acre" if BBS area is in acres
YIELD_FILE = "bbs_yield.csv"

# Normalise names so "Bogura" == "Bogra", "Chattogram" == "Chittagong", etc.
ALIASES = {
    "bogura": "bogra", "barishal": "barisal", "cumilla": "comilla", "jashore": "jessore",
    "chattogram": "chittagong", "chapainawabganj": "nawabganj", "jhalakathi": "jhalokati",
    "jhalakati": "jhalokati", "moulvibazar": "maulvibazar", "netrokona": "netrakona",
    "coxsbazar": "coxsbazar", "brahmanbaria": "brahamanbaria", "gaibandha": "gaibandha",
    "narsingdi": "narsingdi", "sirajganj": "sirajganj", "munshiganj": "munshiganj",
}
def norm(name):
    n = re.sub(r"[^a-z]", "", str(name).lower())
    return ALIASES.get(n, n)

season_df["district_key"] = season_df["district"].map(norm)

if os.path.exists(YIELD_FILE):
    y = pd.read_csv(YIELD_FILE)
    y["district_key"] = y["district"].map(norm)
    if AREA_UNIT == "acre":
        y["area"] = y["area"] / 2.471
    y["yield_t_ha"] = y["production"] / y["area"]

    # data-quality checks on yield
    print("Yield rows:", len(y), "| crops:", y.crop.unique())
    print("Yield outliers (t/ha outside 0.2 - 12):", int(((y.yield_t_ha < 0.2) | (y.yield_t_ha > 12)).sum()))
    unmatched = sorted(set(y.district_key) - set(season_df.district_key))
    print("Yield districts with no climate match (fix names/ALIASES):", unmatched)

    model_df = season_df.merge(
        y[["district_key", "season_year", "crop", "area", "production", "yield_t_ha"]],
        on=["district_key", "season_year", "crop"], how="left")
    print("Rows with a yield value:", int(model_df.yield_t_ha.notna().sum()), "of", len(model_df))
else:
    model_df = season_df.copy()
    print(f"{YIELD_FILE} not found yet. Continuing with climate features only.")

model_df.head()

**Be realistic about yield modelling.** Even in the best case you have at most 64 districts x about 24 seasons per crop (roughly 1,500 rows), and the climate input is coarse. That is enough for exploratory analysis and sanity checks, but not for confident yield forecasts. For the competition, the **transparent suitability score** (climate features compared with cited crop requirements) is the safer main feature, and any yield model should be presented as supporting evidence with its error metrics.

## Step 9: Climatology tables and quick plots

**Climatology** = long-term average for each district and calendar month over 2001-2025. This is exactly what the *Planting-Window Comparison* page needs. It describes **historical patterns, not a forecast**.

In [ ]:
clim = (dm.groupby(["division", "district", "MO"])
          .agg(tmean=("T2M", "mean"), tmax=("T2M_MAX", "mean"), tmin=("T2M_MIN", "mean"),
               dtr=("DTR", "mean"), rh=("RH2M", "mean"), solar=("SOLAR", "mean"),
               rain_mm_mean=("RAIN_MM", "mean"),
               rain_mm_p10=("RAIN_MM", lambda x: x.quantile(.10)),
               rain_mm_p90=("RAIN_MM", lambda x: x.quantile(.90)),
               years=("YEAR", "nunique"))
          .reset_index().round(2))
clim.head()

In [ ]:
# Plot: monthly climatology for 3 districts (edit names to match your district list)
sample = [d for d in ["Dhaka", "Rajshahi", "Sylhet"] if d in clim.district.unique()] or list(clim.district.unique()[:3])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for d in sample:
    c = clim[clim.district == d]
    axes[0].plot(c.MO, c.tmean, marker="o", label=d)
    axes[1].plot(c.MO, c.rain_mm_mean, marker="o", label=d)
axes[0].set_title("Mean temperature (C), 2001-2025 average"); axes[1].set_title("Monthly rainfall (mm), 2001-2025 average")
for a in axes:
    a.set_xlabel("Month"); a.set_xticks(range(1, 13)); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Plot: how variable is Aman-season rainfall across years? (shows climate risk)
d0 = sample[0]
a = season_df[(season_df.crop == "aman_rice") & (season_df.district == d0)].sort_values("season_year")
plt.figure(figsize=(7, 3.5))
plt.bar(a.season_year, a.season_rain_mm)
plt.title(f"Aman season (Jun-Nov) total rainfall, {d0}")
plt.ylabel("mm"); plt.xlabel("Season year"); plt.grid(alpha=.3, axis="y")
plt.show()

## Step 10: Save outputs

| File | Use |
|---|---|
| `district_monthly_climate.csv` | Clean district x year x month table (64 x 25 x 12 rows) |
| `district_climatology.csv` | District x month averages: import into the website database for the planting-window charts |
| `crop_season_features.csv` | District x crop x season-year features (plus yield when available) |

In [ ]:
cols_drop = ["district_key"]
dm.drop(columns=["days"]).round(3).to_csv("district_monthly_climate.csv", index=False)
clim.to_csv("district_climatology.csv", index=False)
model_df.drop(columns=cols_drop).round(3).to_csv("crop_season_features.csv", index=False)
print("Saved 3 files in the Colab file panel (left sidebar, folder icon).")

try:
    from google.colab import files
    for f in ["district_monthly_climate.csv", "district_climatology.csv", "crop_season_features.csv"]:
        files.download(f)
except ImportError:
    pass

## What to do next

1. **Collect and cite crop requirements** (temperature, rainfall, season, water need) for 10 crops from BRRI / BARI / DAE / FAO EcoCrop. Replace the starter thresholds in Step 6.
2. **Build the suitability score** from `district_climatology.csv` and `crop_season_features.csv`: for each crop, score how well a district's month-by-month climate fits the crop's cited requirements, and show per-factor reasons and limitations.
3. **Import the CSVs into your website database** (MySQL) and read them from the crop selector and planting-window pages. Label every chart *"historical data, not a forecast"*.
4. **Optional:** add BBS yields (Step 8) and test a simple baseline model, reporting honest cross-validated error.
5. **Validate:** compare a handful of district values against BMD station records and state the result in your submission's data-quality field.